# FAISS Benchmark Notebook

This notebook benchmarks `embed_ngram` classification with and without FAISS using synthetic data and a deterministic random embedder.

## Setup
Run this notebook from the repository workspace. If you want to exercise the FAISS path, install `faiss-cpu` first.

```bash
pip install -r requirements.txt
pip install faiss-cpu  # optional
```

In [8]:
# Install project requirements before running the benchmark
# This notebook lives under `notebooks/`, so the requirements file is one level up.
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import os
import random
import shutil
import sys
import time

import numpy as np


def bootstrap_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for base in candidates:
        if (base / 'sdg_classifier.py').exists():
            root = base
            break
        if (base / 'ElsevierMapping' / 'sdg_classifier.py').exists():
            root = base / 'ElsevierMapping'
            break
    else:
        root = None
    if root is not None and str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


project_root = bootstrap_project_root()
print('Using project root:', project_root)

from sdg_classifier import SDGClassifier

Using project root: C:\Users\tziou\OneDrive - Cyprus University of Technology\Research\SDGs\ElsevierMapping


In [10]:
def random_embedder_factory(dim=64, seed=0):
    rng = np.random.RandomState(seed)

    def embed(texts):
        out = []
        for text in texts:
            # deterministic pseudo-random vector based on hash
            h = abs(hash(text))
            rng.seed((h + len(text)) % (2 ** 32))
            vector = rng.randn(dim).astype(float)
            vector = vector / (np.linalg.norm(vector) + 1e-12)
            out.append(vector.tolist())
        return out

    return embed

def make_synthetic_sdg_dir(path, n_phrases):
    os.makedirs(path, exist_ok=True)
    phrases = [f'p_{i}' for i in range(n_phrases)]
    with open(os.path.join(path, 'SDG01.txt'), 'w', encoding='utf-8') as handle:
        for phrase in phrases:
            handle.write(f'"{phrase}"\n')
    for i in range(2, 18):
        with open(os.path.join(path, f'SDG{str(i).zfill(2)}.txt'), 'w', encoding='utf-8') as handle:
            handle.write('"unused"\n')

In [14]:
# Benchmark parameters
phrases = 2000
docs = 200
dim = 64
use_faiss = True  # toggle to True if faiss-cpu is installed

tmpdir = Path('tmp_sdg_bench')
if tmpdir.exists():
    shutil.rmtree(tmpdir)
make_synthetic_sdg_dir(str(tmpdir), phrases)
embedder = random_embedder_factory(dim=dim, seed=42)
print(f'Prepared synthetic SDG directory at {tmpdir} with {phrases} phrases.')

Prepared synthetic SDG directory at tmp_sdg_bench with 2000 phrases.


In [12]:
print('Building classifier (embedding phrases)...')
start = time.time()
classifier = SDGClassifier(sdg_dir_path=str(tmpdir), embedder=embedder, use_faiss=use_faiss)
build_time = time.time() - start
print(f'Built in {build_time:.2f}s; phrases loaded: {len(classifier.phrase_index.get(1, []))}')
if use_faiss:
    print('FAISS indices present for SDG1 lengths:', sorted(classifier.faiss_index.get(1, {}).keys()))

Building classifier (embedding phrases)...
Built in 1.91s; phrases loaded: 2000
FAISS indices present for SDG1 lengths: [1]


In [13]:
# Generate synthetic documents that contain one known phrase each
documents = []
for _ in range(docs):
    phrase_index = random.randint(0, phrases - 1)
    documents.append(f'start p_{phrase_index} end')

start = time.time()
for document in documents:
    _ = classifier.classify(None, document, None, similarity_threshold=0.8, similarity_method='embed_ngram')
classify_time = time.time() - start
print(f'Classified {docs} docs in {classify_time:.2f}s ({classify_time / docs:.4f}s/doc)')

Classified 200 docs in 0.19s (0.0009s/doc)


## Notes

- This notebook uses synthetic text and a deterministic random embedder so the benchmark is fast and reproducible.
- Set `use_faiss = True` to benchmark the FAISS-backed path after installing `faiss-cpu`.
- The temporary benchmark directory is created in the workspace as `tmp_sdg_bench/`.